## **PART 1. COLLECTING DATA**

### **1. Topic introduction**

- Health and fitness are important topics globally. Especially in Vietnam, where users are concerned about their
needs.
- Understanding the importance of providing accurate and transparent drug information, our team decided to collect data on products in the functional food and cosmeceutical groups from the Long Chau Pharmacy website, one of the most reputable and popular pharmacy chains in Vietnam. By collecting and analyzing data from this pharmacy's website, we hope to help users easily look up and choose the right drug products for their needs, while ensuring safety. Full and reasonable prices. The data we collect is taken from the source: nhathuoclongchau.com.vn
- In the business field, these data can provide valuable insights into market trends, pricing strategies and consumer preferences, allowing them to optimize services, improve customer satisfaction and maintain competitiveness in the pharmaceutical industry.

### **2. Data description**

The information collected from each product includes:
- **Product Name**: Full name of the product.
- **Category**: Product category group (eg: Skin care, Sexual enhancement, etc.).
- **Price**: Listed price of the product (unit: VND).
- **Trademark**: Brand that manufactures the product.
- **Brand Origin**: Country of the brand.
- **Dosage Form**: Product form (tablet, gel, cream, etc.).
- **Country**: Country where the product is manufactured.
- **Rating**: User rating score.

### **3. Data collection methodology**

The data was collected from the website using the following steps:
1. **Browsing the product list**: Selenium was used to click the "Load More" button and load all the products on the page.

2. **Extracting detailed data**: Information from each product was scraped using BeautifulSoup.

3. **Storing the data**: The data was exported to a CSV file for further processing and analysis.

##### Tools Used:
- **Python**: The main programming language used.
- **Selenium**: Automates web browsing actions.
- **BeautifulSoup**: Parses HTML and extracts data.
- **Pandas**: Stores and processes data in tabular format.

### **4. Source Code and Explanation**

##### a. Import necessary libraries

In [1]:
from bs4 import BeautifulSoup
import requests
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.action_chains import ActionChains
import time

##### b. Functions for Data Extraction

 General Workflow of the Functions
- Input: All functions take a BeautifulSoup object as input, representing the HTML content of the product page.
- Locate the HTML Element Containing the Data: Each function uses BeautifulSoup's search methods (find or find_all) to identify the correct HTML tag containing the required data.
- Handle Exceptions: If the data is not found (due to layout changes or missing values), the function returns "NULL" instead of throwing an error. This ensures the program continues running without interruption.
- Return the Result: The extracted data is returned as a string for further storage or processing.

In [2]:
# Function to get the product title
def get_productName(soup):
    try:
        return soup.find("h1", attrs={'data-test': 'product_name'}).text.strip()
    except (AttributeError, IndexError):
        return "NULL"

# Function to get the product category
def get_category(soup):
    try:
        return soup.find("a", attrs={'class': 'text-blue-500'}).text
    except (AttributeError, IndexError):
        return "NULL"

# Function to get the price information
def get_price(soup):
    try:
        return soup.find("span", attrs={'data-test': 'price'}).text.replace('đ', '').replace('.', '').strip()
    except (AttributeError, IndexError):
        return "NULL"

# Function to get the trademark information
def get_trademark(soup):
    try:
        return soup.find("a", attrs={'class': 'text-blue-5'}).text
    except (AttributeError, IndexError):
        return "NULL"

# Function to get the dosage form
def get_dosageForm(soup):
    try:
        rows = soup.find_all("tr", attrs={'class': 'content-container'})
        for row in rows:
            if "Dạng bào chế" in row.text:  # If "Dosage Form" is found in the row text
                return row.find("div", attrs={'class': 'css-1e2qim1 text-gray-10'}).text.strip()
        return "NULL"  # Return "NULL" if the keyword is not found
    except (AttributeError, IndexError):
        return "NULL"

# Function to get the brand origin
def get_brandOrigin(soup):
    try:
        rows = soup.find_all("tr", attrs={'class': 'content-container'})
        for row in rows:
            if "Xuất xứ thương hiệu" in row.text:  # If "Brand Origin" is found in the row text
                return row.find("div", attrs={'class': 'css-1e2qim1 text-gray-10'}).text.strip()
        return "NULL"
    except (AttributeError, IndexError):
        return "NULL"

# Function to get the country information
def get_country(soup):
    try:
        rows = soup.find_all("tr", attrs={'class': 'content-container'})
        for row in rows:
            if "Nước sản xuất" in row.text:  # If "Country " is found in the row text
                return row.find("div", attrs={'class': 'css-1e2qim1 text-gray-10'}).text.strip()
        return "NULL"
    except (AttributeError, IndexError):
        return "NULL"

# Function to get the rating
def get_rating(soup):
    try:
        return soup.find("span", attrs={'class': 'text-body2 text-gray-7 inline-flex items-center'}).text
    except (AttributeError, IndexError):
        return "NULL"


##### c. Main Function for Web Scraping

The main function combines the individual components and executes the scraping process:
* Browser Automation: Selenium loads the webpage and clicks the "Load More" button repeatedly to reveal all products.
* Data Extraction: BeautifulSoup processes the HTML to retrieve product details using the defined functions.
* Data Storage: Extracted data is saved into a CSV file for analysis.

In [ ]:
if __name__ == '__main__':
    driver = webdriver.Chrome()
    
    # List of URLs you want to scrape data from
    URLs = [
        "https://nhathuoclongchau.com.vn/thuc-pham-chuc-nang",
        "https://nhathuoclongchau.com.vn/duoc-my-pham"
    ]
    
    # Prepare to store product data
    d = {
        "Product Name": [], "Category": [], "Dosage Form": [],
        "Price": [], "Trademark": [], "Brand Origin": [], "Country": [], "Rating": []
    }

    HEADERS = {
        'User-Agent': 'Mozilla/5.0 (Linux; Android 6.0; Nexus 5 Build/MRA58N) AppleWebKit/537.36 '
                      '(KHTML, like Gecko) Chrome/129.0.0.0 Mobile Safari/537.36',
        'Accept-Language': 'en-US, en;q=0.5'
    }
    
    session = requests.Session()
    session.headers.update(HEADERS)

    for URL in URLs:
        driver.get(URL)
        time.sleep(5)

        max_attempts = 400  # Limit the number of clicks to avoid infinite loops
        attempts = 0
        while attempts < max_attempts:
            try:
                show_more_button = WebDriverWait(driver, 10).until(
                    EC.element_to_be_clickable((By.XPATH, '//span[contains(text(), "Xem thêm")]'))
                )
                actions = ActionChains(driver)
                actions.move_to_element(show_more_button).perform()
                show_more_button.click()
                time.sleep(2)
                attempts += 1
            except Exception:
                print("No more products to load or loop limit reached.")
                break
        
        # Get the HTML source after loading
        soup = BeautifulSoup(driver.page_source, "html.parser")

        # Get all product links
        links = soup.find_all("a", attrs={'class': 'block px-3'})
        links_list = ["https://nhathuoclongchau.com.vn" + link.get('href') for link in links]

        # Iterate through the product links and gather data
        for link in links_list:
            try:
                new_webpage = session.get(link)
                new_soup = BeautifulSoup(new_webpage.content, "html.parser")
                d['Product Name'].append(get_productName(new_soup))
                d['Category'].append(get_category(new_soup))
                d['Dosage Form'].append(get_dosageForm(new_soup))
                d['Price'].append(get_price(new_soup))
                d['Trademark'].append(get_trademark(new_soup))
                d['Brand Origin'].append(get_brandOrigin(new_soup))
                d['Country'].append(get_country(new_soup))
                d['Rating'].append(get_rating(new_soup))
            except Exception as e:
                print(f"Error collecting data from {link}: {e}")
                continue
            
    # Close the browser after completion
    driver.quit()

    # Export data to CSV file
    longchau_df = pd.DataFrame.from_dict(d)
    longchau_df.to_csv("data_origin.csv", encoding='utf-8', header=True, index=False)
    print("Data has been saved to data_origin.csv")


### **5. Results**

The data has been saved to the file LongChau_Data.csv. Below is an example of some rows in the dataset:

In [ ]:
df = pd.read_csv("data_origin.csv")
df.head()

### **6. Conclusion**

The data collection has been completed successfully. The data includes detailed information about functional food and cosmetic products from Long Chau Pharmacy.